In [ ]:
!pip install mne

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

from google.colab import drive
drive.mount("/content/drive")

import glob

In [ ]:
set_files = glob.glob(
    "/content/drive/MyDrive/**/sub-*_task-rest_desc-preproc_eeg.set",
    recursive=True
)

set_files = sorted(set_files)

print("Number of EEG files:", len(set_files))

In [ ]:
  parietal_channels = [
    "E106", "E96", "E86", "E87", "E88",
    "E101",
    "E142", "E153", "E162", "E170", "E169"
  ]

  frontal_channels = [
    "E252", "E47", "E48", "E36", "E29",
    "E21",
    "E226", "E2", "E222", "E224", "E5"
  ]


In [ ]:
def analyze_participant(file_path):
  raw = mne.io.read_raw_eeglab(
      file_path,
      preload=True
  )


  participant_id = file_path.split("sub-")[1].split("_task-rest")[0]

  beep_times = []

  for annotation in raw.annotations:
    if annotation["description"] == "Beep":
      beep_times.append(annotation["onset"])

  if len(beep_times) != 4:
    print("Unexpected number of beeps:", len(beep_times))

  eo1 = raw.copy().crop(tmin=beep_times[0], tmax=beep_times[1])
  ec1 = raw.copy().crop(tmin=beep_times[1], tmax=beep_times[2])
  eo2 = raw.copy().crop(tmin=beep_times[2], tmax=beep_times[3])
  ec2 = raw.copy().crop(tmin=beep_times[3], tmax=beep_times[3]+60)

  psd_eo1 = eo1.compute_psd(fmax=50)
  psd_ec1 = ec1.compute_psd(fmax=50)
  psd_eo2 = eo2.compute_psd(fmax=50)
  psd_ec2 = ec2.compute_psd(fmax=50)

  power_eo1 = psd_eo1.get_data()
  power_ec1 = psd_ec1.get_data()
  power_eo2 = psd_eo2.get_data()
  power_ec2 = psd_ec2.get_data()

  freqs = psd_eo1.freqs

  alpha_mask = (freqs >= 9) & (freqs <= 14)
  theta_mask = (freqs >= 4) & (freqs <=8)

  alpha_power_eo1 = power_eo1[:, alpha_mask]
  alpha_power_ec1 = power_ec1[:, alpha_mask]
  alpha_power_eo2 = power_eo2[:, alpha_mask]
  alpha_power_ec2 = power_ec2[:, alpha_mask]

  theta_power_eo1 = power_eo1[:, theta_mask]
  theta_power_ec1 = power_ec1[:, theta_mask]
  theta_power_eo2 = power_eo2[:, theta_mask]
  theta_power_ec2 = power_ec2[:, theta_mask]

  mean_alpha_eo1 = np.mean(alpha_power_eo1, axis=1)
  mean_alpha_ec1 = np.mean(alpha_power_ec1, axis=1)
  mean_alpha_eo2 = np.mean(alpha_power_eo2, axis=1)
  mean_alpha_ec2 = np.mean(alpha_power_ec2, axis=1)

  mean_theta_power_eo1 = np.mean(theta_power_eo1, axis=1)
  mean_theta_power_ec1 = np.mean(theta_power_ec1, axis=1)
  mean_theta_power_eo2 = np.mean(theta_power_eo2, axis=1)
  mean_theta_power_ec2 = np.mean(theta_power_ec2, axis=1)

  mean_alpha_eo = (mean_alpha_eo1 + mean_alpha_eo2) / 2
  mean_alpha_ec = (mean_alpha_ec1 + mean_alpha_ec2) / 2

  mean_theta_eo = (mean_theta_power_eo1 + mean_theta_power_eo2) / 2
  mean_theta_ec = (mean_theta_power_ec1 + mean_theta_power_ec2) / 2

  overall_alpha_eo = np.mean(mean_alpha_eo)
  overall_alpha_ec = np.mean(mean_alpha_ec)

  overall_theta_eo = np.mean(mean_theta_eo)
  overall_theta_ec = np.mean(mean_theta_ec)

  overall_alpha_reactivity = overall_alpha_ec - overall_alpha_eo


  #getting the indices based on raw
  parietal_indices = []
  for i in parietal_channels:
    parietal_indices.append(raw.ch_names.index(i))

  frontal_indices = []
  for i in frontal_channels:
    frontal_indices.append(raw.ch_names.index(i))

  #GETTING THE ALPHA EYES OPEN DATA
  #fancy indexing: Take the mean_alpha_eo, find the values at the parietal_indices, and return those values together in an array

  parietal_alpha_eo = mean_alpha_eo[parietal_indices]
  frontal_alpha_eo = mean_alpha_eo[frontal_indices]

  #GETTING THE MEAN OF THE ALPHA EYES OPEN DATA

  overall_parietal_alpha_eo = np.mean(parietal_alpha_eo)
  overall_frontal_alpha_eo = np.mean(frontal_alpha_eo)


  #GETTING THE ALPHA EYES CLOSED DATA

  parietal_alpha_ec = mean_alpha_ec[parietal_indices]
  frontal_alpha_ec = mean_alpha_ec[frontal_indices]

  #GETTING THE MEAN OF THE ALPHA EYES CLOSED DATA
  overall_parietal_alpha_ec = np.mean(parietal_alpha_ec)


  overall_frontal_alpha_ec = np.mean(frontal_alpha_ec)

  #GETTING THE THETA EYES OPEN DATA

  parietal_theta_eo = mean_theta_eo[parietal_indices]
  frontal_theta_eo = mean_theta_eo[frontal_indices]

  #GETTING THE MEAN OF THETA EYES OPEN DATA

  overall_parietal_theta_eo = np.mean(parietal_theta_eo)


  overall_frontal_theta_eo = np.mean(frontal_theta_eo)


  #GETTING THE THETA EYES CLOSED DATA

  parietal_theta_ec = mean_theta_ec[parietal_indices]
  frontal_theta_ec = mean_theta_ec[frontal_indices]

  #GETTING THE MEAN OF THE THETA EYES CLOSED DATA

  overall_parietal_theta_ec = np.mean(parietal_theta_ec)


  overall_frontal_theta_ec = np.mean(frontal_theta_ec)


  return {
      "participant_id": participant_id,
      "overall_alpha_eo is": overall_alpha_eo,
      "overall_alpha_ec is": overall_alpha_ec,
      "overall_theta_eo is": overall_theta_eo,
      "overall_theta_ec is": overall_theta_ec,
      "overall_alpha_reactivity is": overall_alpha_reactivity,
      "overall_frontal_alpha_ec": overall_frontal_alpha_ec,
      "overall_frontal_alpha_eo": overall_frontal_alpha_eo,
      "overall_parietal_alpha_eo is": overall_parietal_alpha_eo,
      "overall_parietal_alpha_ec is": overall_parietal_alpha_ec,
      "overall_parietal_theta_eo is": overall_parietal_theta_eo,
      "overall_parietal_theta_ec is": overall_parietal_theta_ec,
      "overall_frontal_theta_eo is": overall_frontal_theta_eo,
      "overall_frontal_theta_ec is": overall_frontal_theta_ec,
      "alpha_eo_all_channels": mean_alpha_eo,
      "alpha_ec_all_channels": mean_alpha_ec,
      "theta_eo_all_channels": mean_theta_eo,
      "theta_ec_all_channels": mean_theta_ec
  }

analyze_participant(set_files[0])

results = []
for file_path in set_files:
  results.append(analyze_participant(file_path))








In [ ]:
df = pd.DataFrame(results)

stai_files = glob.glob(
    "/content/drive/MyDrive/**/STAI.tsv",
    recursive=True
)

stai_df = pd.read_csv(stai_files[0], sep="\t")

bis_files = glob.glob(
    "/content/drive/MyDrive/**/BIS.tsv",
    recursive=True
)

bis_df = pd.read_csv(bis_files[0], sep="\t")

stai_df["participant_id"] = stai_df["participant_id"].str.replace("sub-", "")
bis_df["participant_id"] = bis_df["participant_id"].str.replace("sub-","")

merged_df = pd.merge(
    df,
    stai_df,
    on="participant_id"
)

final_merged_df = pd.merge(
    merged_df,
    bis_df,
    on="participant_id"
)



In [ ]:
import pandas as pd

final_merged_df = pd.read_pickle(
    "/content/drive/MyDrive/eeg_final_merged_df.pkl"
)

In [ ]:
eeg_variables = [
    "overall_frontal_alpha_eo",
    "overall_frontal_alpha_ec",
    "overall_parietal_alpha_eo is",
    "overall_parietal_alpha_ec is",
    "overall_frontal_theta_eo is",
    "overall_frontal_theta_ec is",
    "overall_parietal_theta_eo is",
    "overall_parietal_theta_ec is"
]

stai_list = []
bis_list = []

bis_data = final_merged_df.dropna(subset=["bis_total"])

for eeg in eeg_variables:
  r, p = pearsonr(
      final_merged_df["stai_trait"],
      final_merged_df[eeg]
  )

  stai_dictionary = {
      "eeg_variable": eeg,
      "r": r,
      "p": p
  }

  stai_list.append(stai_dictionary)

for eeg in eeg_variables:
  r, p = pearsonr(
      bis_data["bis_total"],
      bis_data[eeg]
  )

  bis_dictionary = {
      "eeg_variable": eeg,
      "r": r,
      "p": p
  }

  bis_list.append(bis_dictionary)


#creating the dataframes
stai_df = pd.DataFrame(stai_list)
bis_df = pd.DataFrame(bis_list)

stai_df["measure"] = "STAI"
bis_df["measure"] = "BIS"

regional_results_df = pd.concat(
  [stai_df, bis_df],
  ignore_index=True
)

regional_results_df

,eeg_variable,r,p,measure
0,overall_frontal_alpha_eo,-0.165829,0.244842,STAI
1,overall_frontal_alpha_ec,-0.007888,0.956189,STAI
2,overall_parietal_alpha_eo is,-0.179032,0.208743,STAI
3,overall_parietal_alpha_ec is,-0.062499,0.663059,STAI
4,overall_frontal_theta_eo is,-0.059291,0.679393,STAI
5,overall_frontal_theta_ec is,-0.123255,0.388860,STAI
6,overall_parietal_theta_eo is,-0.106563,0.456712,STAI
7,overall_parietal_theta_ec is,-0.097088,0.497921,STAI
8,overall_frontal_alpha_eo,-0.047967,0.751578,BIS
9,overall_frontal_alpha_ec,0.217458,0.146580,BIS


In [ ]:
from statsmodels.stats.multitest import multipletests

reject, p_fdr, _, _ = multipletests(
    regional_results_df["p"],
    alpha=0.05,
    method="fdr_bh"
)

regional_results_df["p_fdr"]= p_fdr
regional_results_df["reject"]=reject
print(regional_results_df)

                    eeg_variable         r         p measure     p_fdr  reject
0       overall_frontal_alpha_eo -0.165829  0.244842    STAI  0.933320   False
1       overall_frontal_alpha_ec -0.007888  0.956189    STAI  0.956189   False
2   overall_parietal_alpha_eo is -0.179032  0.208743    STAI  0.933320   False
3   overall_parietal_alpha_ec is -0.062499  0.663059    STAI  0.933320   False
4    overall_frontal_theta_eo is -0.059291  0.679393    STAI  0.933320   False
5    overall_frontal_theta_ec is -0.123255  0.388860    STAI  0.933320   False
6   overall_parietal_theta_eo is -0.106563  0.456712    STAI  0.933320   False
7   overall_parietal_theta_ec is -0.097088  0.497921    STAI  0.933320   False
8       overall_frontal_alpha_eo -0.047967  0.751578     BIS  0.933320   False
9       overall_frontal_alpha_ec  0.217458  0.146580     BIS  0.933320   False
10  overall_parietal_alpha_eo is -0.048209  0.750365     BIS  0.933320   False
11  overall_parietal_alpha_ec is  0.109284  0.469696

In [ ]:
overall_eeg_variables = [
    "overall_alpha_eo is",
    "overall_alpha_ec is",
    "overall_theta_eo is",
    "overall_theta_ec is"
]

overall_stai_list = []
overall_bis_list = []

#Since there are participants without BIS total data this is making a new dataframe without those rows to use for the bis stats
overall_bis_data = final_merged_df.dropna(subset=["bis_total"])

#uses format pearsonr(x, y)
for eeg in overall_eeg_variables:
  r, p = pearsonr(
      final_merged_df["stai_trait"],
      final_merged_df[eeg],
  )

  overall_stai_dictionary = {
      "eeg_variable": eeg,
      "r": r,
      "p": p
  }

  overall_stai_list.append(overall_stai_dictionary)

for eeg in overall_eeg_variables:
  r, p = pearsonr(
      overall_bis_data["bis_total"],
      overall_bis_data[eeg]
  )

  overall_bis_dictionary = {
      "eeg_variable": eeg,
      "r": r,
      "p": p
  }

  overall_bis_list.append(overall_bis_dictionary)


#making these a dataframe
overall_stai_df = pd.DataFrame(overall_stai_list)
overall_bis_df = pd.DataFrame(overall_bis_list)

#This makes a new column in the dataframe, measure, and fills each row with STAI
overall_stai_df["measure"] = "STAI"
#This makes a new column in the dataframe, measure, and fills each row with BIS
overall_bis_df["measure"] = "BIS"

#ignore indec means - Don't keep the old row numbers 0-7 and then 0-7 again - give new df fresh indices 0-15
overall_regional_results_df = pd.concat(
    [overall_stai_df, overall_bis_df],
    ignore_index=True
)

overall_regional_results_df

#FDR correction

from statsmodels.stats.multitest import multipletests

#reject and p_fdr are arrays created!
overall_reject, overall_p_fdr, _, _ = multipletests(
    overall_regional_results_df["p"],
    alpha=0.05,
    method="fdr_bh"
)

overall_regional_results_df["overall_reject"] = overall_reject
overall_regional_results_df["overall_p_fdr"] = overall_p_fdr

print(overall_regional_results_df)

          eeg_variable         r         p measure  overall_reject  \
0  overall_alpha_eo is -0.107654  0.452090    STAI           False   
1  overall_alpha_ec is -0.023576  0.869561    STAI           False   
2  overall_theta_eo is -0.063390  0.658547    STAI           False   
3  overall_theta_ec is -0.052833  0.712718    STAI           False   
4  overall_alpha_eo is  0.007788  0.959032     BIS           False   
5  overall_alpha_ec is  0.137936  0.360632     BIS           False   
6  overall_theta_eo is  0.060565  0.689277     BIS           False   
7  overall_theta_ec is  0.003900  0.979476     BIS           False   

   overall_p_fdr  
0       0.979476  
1       0.979476  
2       0.979476  
3       0.979476  
4       0.979476  
5       0.979476  
6       0.979476  
7       0.979476  


In [ ]:
alpha_eo_matrix = np.stack(final_merged_df["alpha_eo_all_channels"].values)
alpha_eo_matrix.shape


alpha_open_spatial_stai_results = []

for electrode in range(256):
  #this is in [rows, columns] format
  electrode_alpha_eo = alpha_eo_matrix[:, electrode]

  r, p = pearsonr(
      electrode_alpha_eo,
      final_merged_df["stai_trait"]
  )

  alpha_open_spatial_stai_dictionary = {
      "electrode number": electrode + 1,
      "r": r,
      "p": p
  }

  alpha_open_spatial_stai_results.append(alpha_open_spatial_stai_dictionary)

  alpha_open_df_spatial_stai_results = pd.DataFrame(alpha_open_spatial_stai_results)



from statsmodels.stats.multitest import multipletests
alpha_open_spatial_reject, alpha_open_spatial_p_fdr, _, _ = multipletests(
    alpha_open_df_spatial_stai_results["p"],
    alpha=0.05,
    method="fdr_bh"
)

alpha_open_df_spatial_stai_results["reject"] = alpha_open_spatial_reject
alpha_open_df_spatial_stai_results["FDR P"] = alpha_open_spatial_p_fdr

print(alpha_open_df_spatial_stai_results)
alpha_open_df_spatial_stai_results["reject"].sum()

     electrode number         r         p  reject     FDR P
0                   1  0.026762  0.852121   False  0.912397
1                   2 -0.087493  0.541527   False  0.850486
2                   3 -0.189518  0.182865   False  0.703924
3                   4 -0.306373  0.028773   False  0.669253
4                   5 -0.260509  0.064847   False  0.669253
..                ...       ...       ...     ...       ...
251               252  0.020162  0.888321   False  0.924432
252               253 -0.004527  0.974851   False  0.982421
253               254  0.045420  0.751633   False  0.875691
254               255 -0.137547  0.335791   False  0.814259
255               256 -0.071189  0.619598   False  0.850486

[256 rows x 5 columns]


np.int64(0)

In [ ]:
alpha_ec_matrix = np.stack(final_merged_df["alpha_ec_all_channels"].values)
alpha_ec_matrix.shape


alpha_closed_spatial_stai_results = []

for electrode in range(256):
  #this is in [rows, columns] format
  electrode_alpha_ec = alpha_ec_matrix[:, electrode]

  r, p = pearsonr(
      electrode_alpha_ec,
      final_merged_df["stai_trait"]
  )

  alpha_closed_spatial_stai_dictionary = {
      "electrode number": electrode + 1,
      "r": r,
      "p": p
  }

  alpha_closed_spatial_stai_results.append(alpha_closed_spatial_stai_dictionary)

  alpha_closed_df_spatial_stai_results = pd.DataFrame(alpha_closed_spatial_stai_results)



from statsmodels.stats.multitest import multipletests
alpha_closed_spatial_reject, alpha_closed_spatial_p_fdr, _, _ = multipletests(
    alpha_closed_df_spatial_stai_results["p"],
    alpha=0.05,
    method="fdr_bh"
)

alpha_closed_df_spatial_stai_results["reject"] = alpha_closed_spatial_reject
alpha_closed_df_spatial_stai_results["FDR P"] = alpha_closed_spatial_p_fdr

print(alpha_closed_df_spatial_stai_results)
alpha_closed_df_spatial_stai_results["reject"].sum()


     electrode number         r         p  reject     FDR P
0                   1  0.182038  0.201074   False  0.997075
1                   2 -0.073717  0.607186   False  0.997075
2                   3 -0.075612  0.597951   False  0.997075
3                   4 -0.098429  0.491976   False  0.997075
4                   5 -0.026210  0.855136   False  0.997075
..                ...       ...       ...     ...       ...
251               252  0.152634  0.284943   False  0.997075
252               253  0.127211  0.373697   False  0.997075
253               254  0.080393  0.574936   False  0.997075
254               255 -0.033077  0.817763   False  0.997075
255               256 -0.044453  0.756759   False  0.997075

[256 rows x 5 columns]


np.int64(0)

In [ ]:
theta_eo_matrix = np.stack(final_merged_df["theta_eo_all_channels"].values)
theta_eo_matrix.shape


theta_open_spatial_stai_results = []

for electrode in range(256):
  #this is in [rows, columns] format
  electrode_theta_eo = theta_eo_matrix[:, electrode]

  r, p = pearsonr(
      electrode_theta_eo,
      final_merged_df["stai_trait"]
  )

  theta_open_spatial_stai_dictionary = {
      "electrode number": electrode + 1,
      "r": r,
      "p": p
  }

  theta_open_spatial_stai_results.append(theta_open_spatial_stai_dictionary)

  theta_open_df_spatial_stai_results = pd.DataFrame(theta_open_spatial_stai_results)



from statsmodels.stats.multitest import multipletests
theta_open_spatial_reject, theta_open_spatial_p_fdr, _, _ = multipletests(
    theta_open_df_spatial_stai_results["p"],
    alpha=0.05,
    method="fdr_bh"
)

theta_open_df_spatial_stai_results["reject"] = theta_open_spatial_reject
theta_open_df_spatial_stai_results["FDR P"] = theta_open_spatial_p_fdr

print(theta_open_df_spatial_stai_results)
theta_open_df_spatial_stai_results["reject"].sum()


     electrode number         r         p  reject     FDR P
0                   1  0.087302  0.542414   False  0.955424
1                   2 -0.093260  0.515097   False  0.955424
2                   3 -0.093768  0.512804   False  0.955424
3                   4 -0.151839  0.287489   False  0.955424
4                   5 -0.091294  0.524035   False  0.955424
..                ...       ...       ...     ...       ...
251               252  0.102437  0.474426   False  0.955424
252               253  0.048517  0.735294   False  0.955424
253               254  0.060649  0.672463   False  0.955424
254               255 -0.190062  0.181590   False  0.955424
255               256 -0.056176  0.695397   False  0.955424

[256 rows x 5 columns]


np.int64(0)

In [ ]:
theta_ec_matrix = np.stack(final_merged_df["theta_ec_all_channels"].values)
theta_ec_matrix.shape


theta_closed_spatial_stai_results = []

for electrode in range(256):
  #this is in [rows, columns] format
  electrode_theta_ec = theta_ec_matrix[:, electrode]

  r, p = pearsonr(
      electrode_theta_ec,
      final_merged_df["stai_trait"]
  )

  theta_closed_spatial_stai_dictionary = {
      "electrode number": electrode + 1,
      "r": r,
      "p": p
  }

  theta_closed_spatial_stai_results.append(theta_closed_spatial_stai_dictionary)

  theta_closed_df_spatial_stai_results = pd.DataFrame(theta_closed_spatial_stai_results)



from statsmodels.stats.multitest import multipletests
theta_ec_spatial_reject, theta_ec_spatial_p_fdr, _, _ = multipletests(
    theta_closed_df_spatial_stai_results["p"],
    alpha=0.05,
    method="fdr_bh"
)

#once you assign a variable to a dataframe the values in that dataframe stay there even if you change the value of the variable later
theta_closed_df_spatial_stai_results["reject"] = theta_ec_spatial_reject
theta_closed_df_spatial_stai_results["FDR P"] = theta_ec_spatial_p_fdr

print(theta_closed_df_spatial_stai_results)
theta_closed_df_spatial_stai_results["reject"].sum()

     electrode number         r         p  reject     FDR P
0                   1 -0.051836  0.717915   False  0.996034
1                   2 -0.073565  0.607930   False  0.996034
2                   3 -0.066600  0.642403   False  0.996034
3                   4 -0.131055  0.359310   False  0.996034
4                   5 -0.073283  0.609311   False  0.996034
..                ...       ...       ...     ...       ...
251               252  0.071477  0.618178   False  0.996034
252               253 -0.000845  0.995305   False  0.996034
253               254  0.061084  0.670247   False  0.996034
254               255 -0.082829  0.563367   False  0.996034
255               256 -0.072659  0.612369   False  0.996034

[256 rows x 5 columns]


np.int64(0)